# Aliases, formulas and grouped summaries

Name a relationship path with aliasColumn, calculate each row with formulaColumn, and summarize sales by city with group_by.

[Watch lesson 4](https://asqueel.org/learn/04-keys-and-constraints/)

**Run the cells from top to bottom in a fresh Python kernel.** Each notebook includes its own starting state. SQLite runs in memory; restarting the kernel discards the database. To repeat the exercise, restart the kernel and run all cells. Colab execution requires a Google account.

© 2026 Softwell S.r.l. · Apache-2.0

## Prepare the notebook

The video starts in a terminal. Here, install the same official package in the notebook kernel. No virtual environment or separate Python console is needed in Colab.

In [ ]:
%pip install -q "asqueel[migration]==0.6.0"

In [ ]:
%config InteractiveShell.ast_node_interactivity = "all"

## Recreate the starting point

The previous lessons are rebuilt below. The four fictional CSV files are included in this notebook and written into a new temporary folder. No downloads or existing database are needed.

In [ ]:
#@title Prepare the sample CSV files
from pathlib import Path
import os
import tempfile
_lesson_folder = tempfile.TemporaryDirectory(prefix="asqueel-notebook-")
os.chdir(_lesson_folder.name)
_csv_files = {'customers.csv': 'id,name,city\n1,Ada,Florence\n2,Grace,London\n3,Linus,Helsinki\n4,Guido,Amsterdam\n5,Elena,Florence\n6,Alan,Manchester\n7,Margaret,New York\n8,Barbara,Boston\n', 'invoice_rows.csv': 'id,invoice_id,product_code,quantity,unit_price,total\n1,101,TEA,2,3.50,7.00\n2,101,COF,1,6.90,6.90\n3,101,MUG,4,5.25,21.00\n4,102,BTEA,2,3.20,6.40\n5,102,HTEA,3,4.10,12.30\n6,102,HON,4,7.50,30.00\n7,103,SUG,3,2.30,6.90\n8,103,CHO,4,3.90,15.60\n9,103,BIS,1,2.80,2.80\n10,104,JAM,4,4.60,18.40\n11,104,OAT,1,2.40,2.40\n12,104,JUI,2,2.70,5.40\n13,105,WAT,1,0.90,0.90\n14,105,LEM,2,1.80,3.60\n15,105,KET,3,32.90,98.70\n16,106,POT,2,18.50,37.00\n17,106,PLT,3,6.40,19.20\n18,106,BWL,4,7.80,31.20\n19,107,SPO,3,9.50,28.50\n20,107,TRY,4,14.90,59.60\n21,107,JAR,1,8.20,8.20\n22,108,NAP,4,12.00,48.00\n23,108,TWL,1,5.60,5.60\n24,108,FLT,2,2.10,4.20\n25,109,TIN,1,6.70,6.70\n26,109,MAT,2,11.40,22.80\n27,109,CAF,3,29.90,89.70\n28,110,BOT,2,13.50,27.00\n29,110,STR,3,4.80,14.40\n30,110,SCL,4,24.90,99.60\n31,111,TEA,3,3.50,10.50\n32,111,COF,4,6.90,27.60\n33,111,MUG,1,5.25,5.25\n34,112,BTEA,4,3.20,12.80\n35,112,HTEA,1,4.10,4.10\n36,112,HON,2,7.50,15.00\n', 'invoices.csv': 'id,number,customer_id,invoice_date\n101,INV-001,1,2026-09-28\n102,INV-002,2,2026-09-29\n103,INV-003,3,2026-09-30\n104,INV-004,4,2026-10-01\n105,INV-005,5,2026-10-02\n106,INV-006,6,2026-10-03\n107,INV-007,7,2026-10-04\n108,INV-008,8,2026-10-05\n109,INV-009,1,2026-10-06\n110,INV-010,2,2026-10-07\n111,INV-011,5,2026-10-08\n112,INV-012,3,2026-10-09\n', 'products.csv': 'code,name,category,stock\nBTEA,Black tea,Drinks,30\nHTEA,Herbal tea,Drinks,18\nHON,Honey,Food,22\nSUG,Cane sugar,Food,40\nCHO,Dark chocolate,Food,35\nBIS,Biscuits,Food,28\nJAM,Strawberry jam,Food,16\nOAT,Oat drink,Drinks,32\nJUI,Apple juice,Drinks,26\nWAT,Mineral water,Drinks,60\nLEM,Lemonade,Drinks,30\nKET,Kettle,Kitchen,9\nPOT,Teapot,Kitchen,14\nPLT,Plate,Kitchen,36\nBWL,Bowl,Kitchen,24\nSPO,Spoon set,Kitchen,20\nTRY,Serving tray,Kitchen,11\nJAR,Storage jar,Kitchen,25\nNAP,Cotton napkins,Kitchen,18\nTWL,Tea towel,Kitchen,30\nFLT,Coffee filters,Kitchen,45\nTIN,Tea tin,Kitchen,15\nMAT,Table mats,Kitchen,16\nCAF,Coffee maker,Kitchen,7\nBOT,Water bottle,Kitchen,21\nSTR,Tea strainer,Kitchen,23\nSCL,Kitchen scale,Kitchen,8\n'}
for _name, _text in _csv_files.items():
    _ = Path(_name).write_text(_text, encoding="utf-8")


In [ ]:
#@title Recreate the previous lessons
from asqueel import AsqueelDb, SqlDatabaseConfig

model = SqlDatabaseConfig()
root = model.source.db(implementation="sqlite")
_ = root.connection(name=":memory:", implementation="sqlite")

shop = root.packages().package("shop", sqlschema="shop")
product = shop.table("product", pkey="code")
_ = product.column("code", size=":10")
_ = product.column("name", size=":60")

db = AsqueelDb(model)
print(db.migrate().commands)

products = db.table("shop.product")
products.insert({"code": "TEA", "name": "Tea"})
db.commit()

products.query(columns="$code, $name").fetch()

products.update({"name": "Green tea"}, pkey="TEA")
db.commit()

products.query(columns="$code, $name").fetch()
db.commit()

_ = product.column("category", size=":30")
_ = product.column("stock", dtype="I")

db.reloadModel(model)

print(db.migrate().commands)

products = db.table("shop.product")
products.query(columns="$name, $stock").fetch()
db.commit()

products.update({"category": "Drinks", "stock": 8}, pkey="TEA")
db.commit()

products.query(columns="$name, $category, $stock").fetch()
db.commit()

from io import StringIO
data = StringIO(
    "code,name,category,stock\n"
    "COF,Coffee,Drinks,24\n"
    "MUG,Mug,Kitchen,12\n"
)

products.sql_insert(data, format="csv")
db.commit()

products.query(
    columns="$name, $stock",
    where="$category = :category", category="Drinks",
    order_by="$stock DESC",
).fetch()
db.commit()

# Continue with lesson two.
products.query(columns="$code, $name").fetch()
db.commit()

customer = shop.table("customer", pkey="id")
_ = customer.column("id", dtype="I")
_ = customer.column("name", size=":60")
_ = customer.column("city", size=":60")

db.reloadModel(model)
print(db.migrate().commands)

invoice = shop.table("invoice", pkey="id")
_ = invoice.column("id", dtype="I")
_ = invoice.column("number", size=":20")
_ = invoice.column("invoice_date", dtype="D")

customer_id = invoice.column("customer_id", dtype="I")
_ = customer_id.relation("shop.customer.id", foreign_key=True)

invoice_row = shop.table("invoice_row", pkey="id")
_ = invoice_row.column("id", dtype="I")
_ = invoice_row.column("quantity", dtype="I")
_ = invoice_row.column("unit_price", dtype="N", size="12,2")
_ = invoice_row.column("total", dtype="N", size="12,2")

invoice_id = invoice_row.column("invoice_id", dtype="I")
_ = invoice_id.relation("shop.invoice.id", foreign_key=True)
product_code = invoice_row.column("product_code", size=":10")
_ = product_code.relation("shop.product.code", foreign_key=True)

db.reloadModel(model)
print(db.migrate().commands)

products = db.table("shop.product")
customers = db.table("shop.customer")
invoices = db.table("shop.invoice")
rows = db.table("shop.invoice_row")

from pathlib import Path
print("\n".join(Path("invoice_rows.csv").read_text().splitlines()[:4]))

with open("products.csv", newline="", encoding="utf-8") as source:
    products.sql_insert(source)

db.commit()

products.query().count()
db.commit()

with open("customers.csv", newline="", encoding="utf-8") as source:
    customers.sql_insert(source)

db.commit()
customers.query(columns="$name, $city", limit=3).fetch()
db.commit()

with open("invoices.csv", newline="", encoding="utf-8") as source:
    invoices.sql_insert(source)

db.commit()

with open("invoice_rows.csv", newline="", encoding="utf-8") as source:
    rows.sql_insert(source)

db.commit()

invoices.query().count()
rows.query().count()
db.commit()

invoices.query(
    columns="$number, $invoice_date, "
            "@customer_id.name, @customer_id.city",
    limit=3,
).fetch()
db.commit()

rows.query(
    columns="@product_code.name, "
            "$quantity, $unit_price, $total",
    where="$invoice_id = :invoice", invoice=101,
).fetch()
db.commit()

rows.query(
    columns="@invoice_id.number, "
            "@invoice_id.@customer_id.city, "
            "@product_code.name, $total",
    limit=3,
).fetch()
db.commit()

# Continue with lesson three.
invoices.query(
    columns="$number",
    where="@customer_id.city = :city", city="Florence",
).fetch()

invoices.query(
    columns="$number, @customer_id.city",
    order_by="@customer_id.city", limit=4,
).fetch()

rows.query(
    columns="@invoice_id.invoice_date, $total",
    order_by="@invoice_id.invoice_date DESC", limit=3,
).fetch()

rows.query(
    columns="@product_code.name, $quantity",
    where="@invoice_id.@customer_id.city = :city",
    city="Florence", limit=3,
).fetch()

from datetime import date
rows.query(
    columns="@product_code.name, $total",
    where="@invoice_id.@customer_id.city = :city "
          "AND @invoice_id.invoice_date >= :since",
    city="Florence", since=date(2026, 10, 1),
    order_by="@invoice_id.invoice_date DESC", limit=4,
).fetch()

db.commit()


## 1 — What we'll learn in this lesson

Welcome back! Here's what we'll learn in this lesson. Give a relationship path a short name, calculate a value for each invoice row, and group rows into a sales summary by city.

```text
What we'll learn in this lesson
aliasColumn → a short name for a path
formulaColumn → a calculated value per row
group_by → a summary across rows

Same Python session · SQLite in memory
```

The setup above has recreated the previous lessons in this notebook's kernel. Continue with the cells below.

## 2 — Give the city path a short name

Let's call this path city. Alias column gives our two-hop customer city a reusable name. Reload the model and get a fresh table handle.

In [ ]:
_ = invoice_row.aliasColumn(
    "city", relation_path="@invoice_id.@customer_id.city",
)
db.reloadModel(model)
rows = db.table("shop.invoice_row")

Let's read and filter with that name. Dollar city follows the original relationship path. It does not store another copy of the city.

In [ ]:
rows.query(
    columns="$city, @product_code.name",
    where="$city = :city", city="Florence", limit=3,
).fetch()
db.commit()

## 3 — Calculate the amount of each row

Let's calculate quantity times price. Formula column declares an SQL expression. Round it to two decimal places, and describe the result as numeric.

In [ ]:
_ = invoice_row.formulaColumn(
    "calculated_total",
    sql_formula="ROUND($quantity * $unit_price, 2)",
    dtype="N",
)
db.reloadModel(model)
rows = db.table("shop.invoice_row")

Let's read the calculated amounts. Two teas at three fifty make seven. The database evaluates the formula when we query it.

In [ ]:
rows.query(
    columns="$quantity, $unit_price, $calculated_total",
    where="$invoice_id = :invoice", invoice=101,
    order_by="$id",
).fetch()
db.commit()

## 4 — No new stored columns

Let's check whether the database needs changing. True means no migration is needed. The alias and formula change our model, but add no physical columns.

In [ ]:
db.migration_plan().empty

## 5 — Watch the formula respond to a change

Let's temporarily change the quantity to three. The calculated amount becomes ten fifty immediately. The old total imported from CSV stays seven because it is an ordinary stored column.

In [ ]:
rows.update({"quantity": 3}, pkey=1)
rows.query(
    columns="$quantity, $total, $calculated_total",
    where="$id = :id", id=1,
).fetch()

Let's undo that change. Rollback restores the original quantity, so the rest of our examples use the same dataset.

In [ ]:
db.rollback()

## 6 — Summarize sales by city

Now let's summarize many rows. A formula gives one value per invoice row. Group by combines rows that share the same city.

```text
Before: 36 invoice rows
After GROUP BY city: 7 city summaries

SUM(calculated_total) → sales for each city
```

Let's add up the amounts for each city. Group by city, sum the calculated totals, and round the summary to two decimal places. AS sales only names the returned summary column; it is optional.

In [ ]:
rows.query(
    columns="$city, ROUND(SUM($calculated_total), 2) AS sales",
    group_by="$city", order_by="$city",
).fetch()

## 7 — Filter first, then group and sort

Let's find the top cities for drinks. Where filters the invoice rows before grouping. Sum their amounts, sort the sales descending, and show the first three cities.

In [ ]:
rows.query(
    columns="$city, ROUND(SUM($calculated_total), 2) AS sales",
    where="@product_code.category = :category", category="Drinks",
    group_by="$city", order_by="sales DESC", limit=3,
).fetch()
db.commit()

## 8 — Thank you — names, calculations and summaries

Thanks for following along! Use an alias to shorten a path, a formula to calculate each row, and group by to summarize many rows. Next, we will explore virtual relations and their aggregates.

```text
Thank you!
aliasColumn → name a path
formulaColumn → calculate each row
group_by → summarize many rows

Next: virtual relations and related-row aggregates
```

## Check your result

Run this final cell to check the exercise.

In [ ]:
from decimal import Decimal
assert rows.query().count() == 36
amounts = rows.query(columns="$quantity, $unit_price, $calculated_total").fetch()
assert all(Decimal(str(r["calculated_total"])).quantize(Decimal(".01")) == (r["quantity"] * r["unit_price"]).quantize(Decimal(".01")) for r in amounts)
assert rows.query(columns="$quantity", where="$id = :id", id=1).fetch()[0]["quantity"] == 2
report = rows.query(columns="$city, ROUND(SUM($calculated_total), 2) AS sales", group_by="$city").fetch()
expected_sales = {'Florence': '300.65', 'London': '189.70', 'Helsinki': '57.20', 'Amsterdam': '26.20', 'Manchester': '87.40', 'New York': '96.30', 'Boston': '57.80'}
assert {r["city"]: str(Decimal(str(r["sales"])).quantize(Decimal(".01"))) for r in report} == expected_sales
db.commit()
assert db.migration_plan().empty
db.commit()
print("Lesson checks passed.")
